# Iterators and Generators in Python

Goal: understand the iterable/iterator protocol from first principles, build a custom iterator, then see how generators solve the same problem with far less code.

## 1. Iterable vs Iterator — Two Different Things

**Why we need it:** `for x in [1, 2, 3]:` and `for x in "abc":` and `for x in {"a": 1}:` all work, on three completely different data structures. Something uniform is happening underneath that lets `for` treat all of them the same way, and understanding that mechanism is what lets you build your own custom types that work in a `for` loop too.

**Use case:** every single `for` loop you've ever written relies on this distinction, whether you noticed it or not — it's worth naming precisely once, because it explains behavior you'll otherwise find confusing (like why a generator can only be looped over once, later in this notebook).

**How to tell them apart:**
- An **iterable** is anything you *can* loop over — it has an `__iter__` method that produces a fresh iterator, ready to start from the beginning. A `list` is iterable, but a `list` itself doesn't track "where you are" in a loop.
- An **iterator** is the object that actually does the walking — it has a `__next__` method that returns the next value each time it's called, and remembers where it left off. An iterator is also (usually) iterable itself — calling `__iter__` on it just returns itself.

A `list` is iterable but is *not* an iterator — that's precisely why you can loop over the same list twice, in two separate `for` loops, and get all the values both times: each `for` loop asks the list for a brand new iterator to start from scratch.

In [1]:
numbers = [10, 20, 30]

print(hasattr(numbers, "__iter__"))    # a list is iterable...
print(hasattr(numbers, "__next__"))    # ...but not itself an iterator

it = iter(numbers)                      # ask the list for an iterator: calls numbers.__iter__()
print(type(it))                         # a separate object, a "list_iterator"
print(hasattr(it, "__next__"))          # THIS is the iterator -- it has __next__

print(next(it))     # calls it.__next__()
print(next(it))
print(next(it))

try:
    next(it)          # nothing left
except StopIteration:
    print("StopIteration: the iterator is exhausted")

# the original list is completely unaffected -- looping again works fine
for n in numbers:
    print("loop again:", n)

True
False
<class 'list_iterator'>
True
10
20
30
StopIteration: the iterator is exhausted
loop again: 10
loop again: 20
loop again: 30


## 2. What `for` Actually Does

**Why we need it:** `for x in numbers:` looks like a single primitive operation, but it's really just a convenient shorthand for exactly the three lines of manual `iter()`/`next()`/`StopIteration` handling from the cell above. Seeing that expansion is what makes it obvious how to make your *own* class work in a `for` loop later in this notebook — you just need to make it behave the way `for` already expects.

**Use case:** understanding this is what turns "for loops are magic" into "for loops are calling two methods I could implement myself" — the prerequisite for writing a custom iterator.

**How it works:** `for x in something:` is equivalent to calling `iter(something)` once to get an iterator, then calling `next()` on it repeatedly, assigning each result to `x`, until `next()` raises `StopIteration` — at which point the loop ends cleanly, without that exception ever becoming visible to your code.

In [2]:
numbers = [10, 20, 30]

# what "for n in numbers:" really does, written out manually
manual_iterator = iter(numbers)
while True:
    try:
        n = next(manual_iterator)
    except StopIteration:
        break
    print(n)

print("--- vs the real thing ---")
for n in numbers:      # identical result, but Python handles iter()/next()/StopIteration for you
    print(n)

10
20
30
--- vs the real thing ---
10
20
30


## 3. Building a Custom Iterator

**Why we need it:** sometimes the sequence you want to loop over doesn't already exist as a `list` — it needs to be *computed*, value by value, and computing every value up front (into a big list) would waste memory or time you don't need to spend, especially if the loop might stop early. A custom iterator computes each value only when it's actually asked for.

**Use case:** generating the first N Fibonacci numbers without pre-building the entire list in memory first — useful if N is huge, or if the calling code might `break` out of the loop early and never need most of the values at all.

**How to implement:** write a class with both methods `for` expects: `__iter__(self)`, which usually just `return self` (the object tracks its own position, so it can serve as its own iterator), and `__next__(self)`, which computes and returns the next value, raising `StopIteration` once there's nothing left to produce.

In [3]:
class FibonacciIterator:
    def __init__(self, count):
        self.count = count      # how many numbers to produce
        self.produced = 0       # how many we've produced so far
        self.a, self.b = 0, 1   # the running pair used to compute each next value

    def __iter__(self):
        return self

    def __next__(self):
        if self.produced >= self.count:
            raise StopIteration
        value = self.a
        self.a, self.b = self.b, self.a + self.b   # advance to the next pair
        self.produced += 1
        return value


for num in FibonacciIterator(8):
    print(num)

# it really is a proper iterator: iter()/next() work on it directly too
fib = FibonacciIterator(3)
print(next(fib), next(fib), next(fib))
try:
    next(fib)
except StopIteration:
    print("StopIteration: only asked for 3 numbers")

0
1
1
2
3
5
8
13
0 1 1
StopIteration: only asked for 3 numbers


## 4. The Problem: Custom Iterators Are a Lot of Boilerplate

**Why we need it:** look back at `FibonacciIterator` — three `__init__` fields just to track *where we are* (`produced`, `a`, `b`), plus manual `StopIteration` logic, to express an idea that's genuinely simple: "start at 0 and 1, keep adding, stop after N numbers." Most of that class is bookkeeping, not the actual logic. As the sequence gets more complex, the bookkeeping grows faster than the logic does.

**Use case:** literally every case from section 3 — anywhere you want lazy, on-demand values — but without needing to manually manage "where am I" state by hand.

**How Python solves it:** a **generator**, which is a regular-looking function that produces a sequence of values using `yield` instead of `return`, and gets the entire `__iter__`/`__next__`/`StopIteration` machinery for free, generated automatically by Python. The next sections show what that actually looks like.

## 5. Generator Functions and `yield`

**Why we need it:** to rewrite `FibonacciIterator` as a few lines of straightforward, linear-looking code, while keeping the exact same lazy, one-value-at-a-time behavior.

**Use case:** any of the situations from section 3 — but now written as plain, readable logic instead of a class with manually tracked state.

**How to implement:** write a normal `def`, but use `yield value` instead of `return value` anywhere you'd produce an item. The moment a function body contains `yield` anywhere, Python treats it specially: calling the function doesn't run the code at all — it immediately returns a **generator object** (already a valid iterator, no extra class needed). Each call to `next()` on that generator runs the function's code *only up to the next `yield`*, hands back that value, and then **pauses**, remembering exactly where it stopped — including every local variable's value. The next `next()` call resumes right after that `yield` and continues until the following one. When the function finally returns (falls off the end), Python automatically raises `StopIteration` for you — you never write that yourself.

In [4]:
def fibonacci(count):
    a, b = 0, 1
    for _ in range(count):
        yield a                  # pause here, hand back `a`, remember everything, resume on next()
        a, b = b, a + b


gen = fibonacci(8)
print(type(gen))                 # a generator object -- already an iterator, no class was written

for num in gen:                  # works directly in a for-loop, same as FibonacciIterator did
    print(num)

# proving the pause-and-resume behavior with manual next() calls
g = fibonacci(3)
print("calling next() manually:")
print(next(g))    # runs up to the first yield
print(next(g))    # resumes right after that yield, runs to the next one
print(next(g))
try:
    next(g)         # the function has now returned -- StopIteration, automatically
except StopIteration:
    print("StopIteration: generator is exhausted, just like the manual iterator was")

<class 'generator'>
0
1
1
2
3
5
8
13
calling next() manually:
0
1
1
StopIteration: generator is exhausted, just like the manual iterator was


## 6. A Gotcha: Generators Are Single-Use

**Why we need it:** section 1 made a point of showing that a `list` can be looped over twice, in two separate `for` loops, because each loop asks it for a *fresh* iterator. A generator object is different — since it *is* the iterator itself (like `FibonacciIterator` from section 3, which also returned `self` from `__iter__`), once it's been exhausted, there's no "resetting" it. Looping over the same generator object a second time silently produces nothing at all — no error, which makes this an easy mistake to miss.

**Use case:** this matters the moment you try to reuse a generator variable, e.g. passing the same generator to two different functions that each expect to loop over all the values — only the first one will actually get any.

**How to work around it:** if you need to iterate multiple times, call the generator *function* again each time (`fibonacci(8)` creates a brand new generator object), or convert it to a `list` once if you genuinely need to keep all the values around (accepting the memory cost that generators exist to avoid).

In [5]:
gen = fibonacci(5)

print("first loop:")
for num in gen:
    print(num)

print("second loop over the SAME generator object:")
for num in gen:
    print(num)          # nothing prints -- already exhausted, no error raised

print("calling the function again creates a fresh generator:")
for num in fibonacci(5):
    print(num)          # works fine -- this is a brand new generator object

first loop:
0
1
1
2
3
second loop over the SAME generator object:
calling the function again creates a fresh generator:
0
1
1
2
3


## 7. Generator Expressions and Lazy Evaluation

**Why we need it:** a list comprehension like `[x * x for x in range(1_000_000)]` builds the *entire* million-element list in memory immediately, even if you only end up using the first few values or summing them all without ever needing them stored at once. That's often wasted memory for no benefit.

**Use case:** feeding a large or expensive computation straight into something like `sum()` or a `for` loop, where you never actually need all the values to exist in memory simultaneously — only one at a time, as they're consumed.

**How to implement:** swap the square brackets of a list comprehension for parentheses: `(x * x for x in range(1_000_000))`. This is a **generator expression** — syntactically almost identical to a list comprehension, but it produces values lazily, one at a time, exactly like a generator function does, without needing `def`/`yield` at all for simple cases.

In [6]:
import sys

list_comp = [x * x for x in range(1_000_000)]     # built entirely, right now, in memory
gen_expr = (x * x for x in range(1_000_000))       # nothing computed yet -- just a plan

print(type(gen_expr))
print(f"list takes about {sys.getsizeof(list_comp):,} bytes")
print(f"generator takes about {sys.getsizeof(gen_expr):,} bytes")   # tiny, regardless of range size

print(sum(list_comp))                                # sums the already-built list
print(sum(x * x for x in range(1_000_000)))          # sums values one at a time, never stores them all

# a generator expression can be passed straight into a function without extra parentheses
print(any(x > 999_990 for x in range(1_000_000)))    # stops as soon as it finds a match -- lazy

<class 'generator'>
list takes about 8,448,728 bytes
generator takes about 200 bytes
333332833333500000
333332833333500000
True


## 8. Infinite Generators

**Why we need it:** every version of `fibonacci` so far took a `count` and stopped. But "the sequence of all Fibonacci numbers" is conceptually infinite — there's no natural place to stop baked into the idea itself. A `list` genuinely *cannot* represent that (it would need infinite memory), but a generator can, because it only ever computes one value at a time and never needs to hold the rest.

**Use case:** a stream of values with no natural endpoint — an infinite counter, infinite Fibonacci numbers, reading lines from a socket that stays open — combined with logic elsewhere that decides when to stop asking for more.

**How to implement:** write a generator function with a `while True:` loop (or no loop-ending condition at all) around the `yield`. It's only "safe" because nothing forces the generator to run to completion — it truly only computes the next value when something calls `next()` on it. Pair it with `itertools.islice` (or a manual counter and `break`) to take just the first N values from what is, in principle, an endless sequence.

In [7]:
from itertools import islice


def infinite_fibonacci():
    a, b = 0, 1
    while True:            # no stopping condition -- runs forever, in principle
        yield a
        a, b = b, a + b


gen = infinite_fibonacci()
first_ten = list(islice(gen, 10))   # islice only pulls 10 values -- the rest are never computed
print(first_ten)

# manual version of the same idea, without itertools
gen2 = infinite_fibonacci()
values = []
for num in gen2:
    if num > 100:
        break               # stop asking for more -- the generator itself never "finishes"
    values.append(num)
print(values)

[0, 1, 1, 2, 3, 5, 8, 13, 21, 34]
[0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89]


## 9. Putting It Together: Processing a Large Log File, Lazily

**Why we need it:** to see why all of this matters in a realistic setting, not just number sequences — processing data that's too large (or too slow to produce) to comfortably hold in memory all at once.

**Use case:** scanning a log file for error lines and reporting only the first 3 matches, without reading the entire file into memory first or scanning past the point where you already have what you need — exactly the scenario generators are built for.

**How it's implemented here:** `read_lines` is a generator that yields log lines one at a time (standing in for reading a real file line-by-line, which is itself lazy in Python). `error_lines` is a generator expression built on top of it, filtering for `"ERROR"` without producing a separate stored list. `islice` pulls only the first 3 matches — the generator chain stops being pulled from immediately after that, so log lines past the 3rd error are never even processed.

In [8]:
def read_lines():
    """Stands in for `for line in open('huge.log'):` -- also lazy in real Python."""
    lines = [
        "INFO: server started",
        "INFO: handling request /home",
        "ERROR: database timeout on request /checkout",
        "INFO: handling request /about",
        "ERROR: null pointer in payment handler",
        "INFO: handling request /profile",
        "ERROR: disk write failed for session log",
        "INFO: handling request /contact",   # never reached below -- we stop after 3 errors
        "ERROR: this one is never even processed",
    ]
    for line in lines:
        print(f"  (reading: {line!r})")   # proves how far into the file we actually read
        yield line


error_lines = (line for line in read_lines() if line.startswith("ERROR"))

print("first 3 error lines:")
for error in islice(error_lines, 3):
    print(error)

first 3 error lines:
  (reading: 'INFO: server started')
  (reading: 'INFO: handling request /home')
  (reading: 'ERROR: database timeout on request /checkout')
ERROR: database timeout on request /checkout
  (reading: 'INFO: handling request /about')
  (reading: 'ERROR: null pointer in payment handler')
ERROR: null pointer in payment handler
  (reading: 'INFO: handling request /profile')
  (reading: 'ERROR: disk write failed for session log')
ERROR: disk write failed for session log
